In [ ]:
from pathlib import Path
import importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from functions.run_config import RunConfig
from functions.pipeline import prepare_inputs
from functions.metadata import load_beskrivelser_events

import functions.variable_event_nonevent as fastprog
importlib.reload(fastprog)

from functions.variable_event_nonevent import (
    build_episodes_fast,
    DEFAULT_PRE_ANCHOR_FEATURES,
    save_feature_name_table,
)

# ============================================================
# PATHS
# ============================================================

BASE_DIR = Path(".").resolve()
INPUT_DIR = Path("simulation_files")
beskrivelser_file = Path("beskrivelser_final2.xlsx")

cfg = RunConfig(base_dir=BASE_DIR, input_dir=INPUT_DIR, tr="1H")
cfg.ensure_dirs()

simres_paths, param_paths, valid = prepare_inputs(
    cfg,
    beskrivelser_files=[beskrivelser_file],
)

print("Valid events:", len(valid))
print("Unique GUIDs:", valid["GUID"].nunique())

# ============================================================
# SETTINGS
# ============================================================

WINDOW_BEFORE_H = 72
WINDOW_AFTER_H = 72
LAGS_H = [3, 6, 12, 24, 48, 72]

TOL_MINUTES = 30

GW_DEF = "gw"
GW_SM_WEIGHT = 1.0

GW_ACT_Q = 0.99
Q_ACT_Q = 0.99

SEASON_DAYS = 30
EXCLUDE_EVENT_BUFFER_DAYS = 14

N_CONTROLS = 25
DECLUSTER_DAYS = 7
MIN_CONTROLS_PER_YEAR = 2
RANDOM_SEED = 123

MATCH_TARGET = "gw"
# Options:
# "gw", "prsum", "q", "both", "all3", "both_qweighted"

MATCH_MODE = "value_pct"
# Options:
# "value_pct", "quantile"

Q_WEIGHT = 0.20
# Only used for MATCH_TARGET = "both_qweighted"

VALUE_TOLERANCE_LEVELS = [0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30]
QUANTILE_TOLERANCE_LEVELS = [0.005, 0.01, 0.02, 0.05, 0.10]

PRSUM_LAG_H = 24

VAR_FEATURES = DEFAULT_PRE_ANCHOR_FEATURES.copy()

# ============================================================
# RUN
# ============================================================

episodes_df, non_event_anchors_df, non_event_anchors = build_episodes_fast(
    valid=valid,
    simres_paths=simres_paths,
    param_paths=param_paths,

    gw_def=GW_DEF,
    gw_sm_weight=GW_SM_WEIGHT,

    window_before_h=WINDOW_BEFORE_H,
    window_after_h=WINDOW_AFTER_H,
    lags_h=LAGS_H,
    tol_minutes=TOL_MINUTES,

    season_days=SEASON_DAYS,
    exclude_event_buffer_days=EXCLUDE_EVENT_BUFFER_DAYS,
    n_controls=N_CONTROLS,

    match_target=MATCH_TARGET,
    match_mode=MATCH_MODE,
    value_tolerance_levels=VALUE_TOLERANCE_LEVELS,
    quantile_tolerance_levels=QUANTILE_TOLERANCE_LEVELS,
    prsum_lag_h=PRSUM_LAG_H,

    q_weight=Q_WEIGHT,

    decluster_days=DECLUSTER_DAYS,
    min_controls_per_year=MIN_CONTROLS_PER_YEAR,
    random_seed=RANDOM_SEED,

    gw_act_q=GW_ACT_Q,
    q_act_q=Q_ACT_Q,

    var_features=VAR_FEATURES,
)

print("Episodes total:", len(episodes_df))
print(episodes_df["y"].value_counts(dropna=False))

# ============================================================
# SAVE OUTPUTS
# ============================================================

suffix = f"{MATCH_TARGET}_{MATCH_MODE}_pr{PRSUM_LAG_H}h"

out_csv = cfg.results_dir / f"{cfg.input_dir.name}_episodes_fast_progressive_{suffix}ikkeskred.csv"
anchors_csv = cfg.results_dir / f"{cfg.input_dir.name}_non_event_anchors_fast_progressive_{suffix}.csv"
feature_table_csv = cfg.results_dir / f"{cfg.input_dir.name}_feature_description_{suffix}.csv"

episodes_df.to_csv(out_csv, index=False)
non_event_anchors_df.to_csv(anchors_csv, index=False)
save_feature_name_table(feature_table_csv, lags_h=LAGS_H)

print("Saved episodes:", out_csv)
print("Saved anchors:", anchors_csv)
print("Saved feature table:", feature_table_csv)

display(non_event_anchors_df.head())
display(episodes_df.head())

# ============================================================
# QUICK SANITY SUMMARIES
# ============================================================

print("Tier distribution:")
display(
    non_event_anchors_df
    .groupby(["match_tier", "match_tolerance_used"], dropna=False)
    .size()
    .rename("n")
    .reset_index()
    .sort_values("match_tier")
)

print("Worst matches:")
display(
    non_event_anchors_df
    .sort_values("match_distance", ascending=False)
    .head(10)
)